# 01 — Training Iniz Agent Guard (3-head classifier)

Melatih **Qwen2.5-0.5B + LoRA r=8** dengan tiga head:
`injection_score` (regresi), `shell_risk_score` (regresi, target PROXY),
`action` (klasifikasi 4 kelas).

Notebook ini adalah skrip yang **benar-benar menghasilkan** `checkpoint-2634` —
sudah diverifikasi: 488 kunci state_dict identik dengan checkpoint dan 11/11
hyperparameter cocok dengan `training_args.bin`
(lihat `scripts/verify_arch_match.py`).

Setelah training selesai → lanjut ke `02_verify_export_deploy.ipynb`.

## ⚠️ RESTART KERNEL sebelum menjalankan notebook ini

`CUDA_VISIBLE_DEVICES` harus diset **sebelum** torch di-import di mana pun dalam
proses ini.

## Enam bug yang sudah diperbaiki di sini

| # | Bug | Perbaikan |
|---|---|---|
| 1 | CUDA OOM karena Trainer membungkus model dengan DataParallel begitu >1 GPU terlihat | `CUDA_VISIBLE_DEVICES=0` sebelum import torch |
| 2 | `RuntimeError: mat1 and mat2 must have same dtype` (pooled BF16 vs head FP32) | head dibuat dengan `MODEL_DTYPE` yang sama + `.to(dtype=...)` di forward |
| 3 | `gradient_checkpointing_enable` tidak ada di `torch.nn.Module` kustom | implementasi delegasi ke `self.base` |
| 4 | Overhead memori `AutoModelForCausalLM` (lm_head 151936×896) | pakai `AutoModel` (base) — tidak butuh lm_head |
| 5 | Overhead `output_hidden_states=True` menyimpan hidden state semua layer | jangan set; pakai `outputs.last_hidden_state` |
| 6 | Pooling mengambil **padding token** karena `padding="max_length"` + `[:, -1, :]` | ambil indeks `attention_mask.sum(1) - 1` |
| 7 | `Expected all tensors to be on the same device` di pre-flight test | `guard_model.cuda()` sebelum forward manual (Trainer memindah otomatis, forward manual tidak) |


In [ ]:
# ============================================================
# 0. LIMIT PYTORCH KE SATU GPU — WAJIB sebelum import torch
# ============================================================
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("CUDA_VISIBLE_DEVICES =", os.environ["CUDA_VISIBLE_DEVICES"])

In [ ]:
# ============================================================
# 1. INSTALL
# ============================================================
%pip install -q transformers datasets peft accelerate optimum[openvino] nncf "torchao>=0.16.0" --break-system-packages

In [ ]:
# ============================================================
# 2. IMPORT
# ============================================================
import gc
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

In [ ]:
# ============================================================
# 3. ENVIRONMENT CHECK
# ============================================================
print("=" * 70)
print("ENVIRONMENT")
print("=" * 70)
print("PyTorch          :", torch.__version__)
print("CUDA available   :", torch.cuda.is_available())
print("Visible GPU count:", torch.cuda.device_count())

if torch.cuda.device_count() > 1:
    print("\n!! >1 GPU terlihat — RESTART KERNEL dan jalankan ulang dari cell 0.")

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}  VRAM {p.total_memory / 1024**3:.2f} GB")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 4. Dataset & label mapping

Mapping ini adalah **satu sumber kebenaran** dan sudah divalidasi dengan model
sebagai hakim: mapping benar memberi accuracy 0,966 vs 0,741 untuk mapping salah
pada model yang sama (`scripts/verify_incoming_mapping.py`).

`shell_risk_score` adalah **proxy keyword**, bukan label ground-truth — dataset ini
tidak punya contoh shell-injection sungguhan. Baca hasil head shell dengan
skeptisisme lebih besar dari head injection.


In [ ]:
# ============================================================
# 5. DATASET
# ============================================================
print("=" * 70)
print("LOADING DATASET")
print("=" * 70)

ds_raw = load_dataset("neuralchemy/Prompt-injection-dataset", "full")
train = ds_raw["train"].to_pandas()
val = ds_raw["validation"].to_pandas()
test = ds_raw["test"].to_pandas()

print(f"Train     : {len(train):,}")
print(f"Validation: {len(val):,}")
print(f"Test      : {len(test):,}")
print()
print(train["category"].value_counts())

In [ ]:
# ============================================================
# 6. LABEL MAPPING
# ============================================================
SEVERITY_MAP = {"low": 0.3, "medium": 0.5, "high": 0.7, "critical": 0.9}

OVERRIDE_CATEGORIES = {
    "direct_injection", "jailbreak", "persona_replacement",
    "many_shot", "crescendo",
}
OBFUSCATION_CATEGORIES = {
    "encoding_obfuscation", "token_smuggling",
    "indirect_injection", "context_overflow",
}
EXTRACTION_CATEGORIES = {"system_extraction", "prompt_leaking"}

SHELL_RISK_KEYWORDS = [
    "eval", "exec", "atob", "bash", "shell", "subprocess",
    "os.system", "js_eval", "code_execution", "command", "__import__",
]


def compute_shell_risk_proxy(row):
    """PROXY keyword, bukan label asli. Dataset tidak punya contoh shell nyata."""
    if int(row["label"]) == 0:
        return 0.0

    raw_tags = row["tags"]
    if raw_tags is None:
        tag_list = []
    else:
        try:
            tag_list = [str(x) for x in list(raw_tags)]
        except Exception:
            tag_list = [str(raw_tags)]

    haystack = (str(row["text"]) + " " + " ".join(tag_list)).lower()
    if any(kw in haystack for kw in SHELL_RISK_KEYWORDS):
        return 0.6
    return 0.1


def map_labels(row):
    category = row["category"]
    severity = row["severity"]
    label = int(row["label"])

    injection = 0.0
    action = "PASS"

    if label == 1:
        injection = SEVERITY_MAP.get(severity, 0.5)
        if category in OVERRIDE_CATEGORIES:
            action = "PAUSE_AGENTS"
        elif category in OBFUSCATION_CATEGORIES:
            action = "ISOLATE_FILE"
        elif category in EXTRACTION_CATEGORIES:
            action = "USER_CONFIRMATION"
        else:
            action = "PAUSE_AGENTS"

    return pd.Series({
        "inj": injection,
        "shell": compute_shell_risk_proxy(row),
        "action": action,
    })


print("Generating labels...")
for df in (train, val, test):
    df[["inj", "shell", "action"]] = df.apply(map_labels, axis=1)

print("\nAction distribution (train):")
print(train["action"].value_counts())

In [ ]:
# ============================================================
# 7. ACTION LABELS + KONFIG
# ============================================================
ACTIONS = ["PASS", "PAUSE_AGENTS", "ISOLATE_FILE", "USER_CONFIRMATION"]
ACTION2IDX = {a: i for i, a in enumerate(ACTIONS)}
IDX2ACTION = {i: a for a, i in ACTION2IDX.items()}

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

# PENTING: MAX_LENGTH ini menentukan seq_len IR saat export.
# Export WAJIB memakai --seq-len 128 agar cocok.
MAX_LENGTH = 128

print("Action mapping:", ACTION2IDX)
print("MAX_LENGTH    :", MAX_LENGTH, "(export IR harus --seq-len 128)")

In [ ]:
# ============================================================
# 8. TOKENIZER
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

## 9. Dataset class & collator

Perhatikan `padding="max_length"`: ini yang membuat pooling `[:, -1, :]` naif
mengambil **padding token**. Pooling yang benar ada di `forward()` cell 12.


In [ ]:
# ============================================================
# 10. DATASET CLASS
# ============================================================
class GuardDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=128):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        encoded = self.tokenizer(
            str(row["text"]),
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": encoded["input_ids"][0],
            "attention_mask": encoded["attention_mask"][0],
            "inj_target": torch.tensor(float(row["inj"]), dtype=torch.float32),
            "shell_target": torch.tensor(float(row["shell"]), dtype=torch.float32),
            "action_target": torch.tensor(ACTION2IDX[row["action"]], dtype=torch.long),
        }


def guard_collator(batch):
    return {
        "input_ids": torch.stack([x["input_ids"] for x in batch]),
        "attention_mask": torch.stack([x["attention_mask"] for x in batch]),
        "inj_target": torch.stack([x["inj_target"] for x in batch]),
        "shell_target": torch.stack([x["shell_target"] for x in batch]),
        "action_target": torch.stack([x["action_target"] for x in batch]),
    }


train_ds = GuardDataset(train, tokenizer, MAX_LENGTH)
val_ds = GuardDataset(val, tokenizer, MAX_LENGTH)
test_ds = GuardDataset(test, tokenizer, MAX_LENGTH)

print("=" * 70)
print("DATASET READY")
print("=" * 70)
print("Train:", len(train_ds))
print("Val  :", len(val_ds))
print("Test :", len(test_ds))

In [ ]:
# ============================================================
# 11. DTYPE + BACKBONE + LoRA
#
# AutoModel, BUKAN AutoModelForCausalLM — lm_head (151936x896) tidak
# dibutuhkan dan hanya membuang memori. last_hidden_state VALID di
# AutoModel base (mengembalikan BaseModelOutputWithPast).
# ============================================================
if torch.cuda.is_available():
    MODEL_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    MODEL_DTYPE = torch.float32
print("Model dtype:", MODEL_DTYPE)

print("\nLoading Qwen backbone (AutoModel)...")
backbone = AutoModel.from_pretrained(BASE_MODEL, torch_dtype=MODEL_DTYPE)
if hasattr(backbone.config, "use_cache"):
    backbone.config.use_cache = False
print("Hidden size:", backbone.config.hidden_size)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="FEATURE_EXTRACTION",
    bias="none",
)
backbone = get_peft_model(backbone, lora_config)
backbone.print_trainable_parameters()

## 12. Model 3-head

Tiga detail yang penting dan mudah salah:

1. **dtype head == dtype backbone.** Kalau head dibuat FP32 sementara `pooled` BF16,
   hasilnya `RuntimeError: mat1 and mat2 must have same dtype`.
2. **Pooling ambil last valid token**, bukan `[:, -1, :]` — karena
   `padding="max_length"` membuat posisi terakhir sering padding.
3. **Loss dihitung FP32** untuk stabilitas numerik, walau forward BF16.


In [ ]:
# ============================================================
# 13. GUARD HEAD MODEL
# ============================================================
class GuardHeadModel(torch.nn.Module):
    def __init__(self, base, actions_dim):
        super().__init__()
        self.base = base
        hidden_size = base.config.hidden_size

        # FIX: head memakai dtype yang SAMA dengan backbone.
        self.inj_head = torch.nn.Linear(hidden_size, 1).to(dtype=MODEL_DTYPE)
        self.shell_head = torch.nn.Linear(hidden_size, 1).to(dtype=MODEL_DTYPE)
        self.action_head = torch.nn.Linear(hidden_size, actions_dim).to(dtype=MODEL_DTYPE)

        self.mse = torch.nn.MSELoss()
        self.ce = torch.nn.CrossEntropyLoss()

    # FIX: Trainer memanggil ini; torch.nn.Module kustom tidak punya secara default.
    def gradient_checkpointing_enable(self, gradient_checkpointing_kwargs=None):
        if hasattr(self.base, "gradient_checkpointing_enable"):
            if gradient_checkpointing_kwargs is None:
                self.base.gradient_checkpointing_enable()
            else:
                self.base.gradient_checkpointing_enable(
                    gradient_checkpointing_kwargs=gradient_checkpointing_kwargs
                )

    def gradient_checkpointing_disable(self):
        if hasattr(self.base, "gradient_checkpointing_disable"):
            self.base.gradient_checkpointing_disable()

    def forward(self, input_ids, attention_mask,
                inj_target=None, shell_target=None, action_target=None):
        # JANGAN set output_hidden_states=True — menyimpan hidden state SEMUA layer.
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state

        # FIX: ambil LAST VALID token, bukan [:, -1, :] yang bisa kena padding.
        last_indices = attention_mask.sum(dim=1) - 1
        batch_indices = torch.arange(hidden_states.size(0), device=hidden_states.device)
        pooled = hidden_states[batch_indices, last_indices]
        pooled = pooled.to(dtype=self.inj_head.weight.dtype)

        injection_logits = self.inj_head(pooled).squeeze(-1)
        shell_logits = self.shell_head(pooled).squeeze(-1)
        action_logits = self.action_head(pooled)

        loss = None
        if inj_target is not None and shell_target is not None and action_target is not None:
            loss = (
                self.mse(injection_logits.float(), inj_target.float())
                + self.mse(shell_logits.float(), shell_target.float())
                + self.ce(action_logits.float(), action_target)
            )

        return {
            "loss": loss,
            "injection": injection_logits,
            "shell": shell_logits,
            "action": action_logits,
        }


guard_model = GuardHeadModel(backbone, len(ACTIONS))
guard_model.gradient_checkpointing_enable()

total = sum(p.numel() for p in guard_model.parameters())
trainable = sum(p.numel() for p in guard_model.parameters() if p.requires_grad)
print("=" * 70)
print(f"Total      : {total:,}")
print(f"Trainable  : {trainable:,}")
print(f"Trainable %: {100 * trainable / total:.4f}%")
print()
print("Backbone dtype  :", next(guard_model.base.parameters()).dtype)
print("Head dtypes     :", guard_model.inj_head.weight.dtype,
      guard_model.shell_head.weight.dtype, guard_model.action_head.weight.dtype)

## 14. Pre-flight forward test

Dijalankan **sebelum** `trainer.train()` supaya error dtype/device muncul dalam 2
detik, bukan setelah menunggu training mulai.

`guard_model.cuda()` di sini **wajib**: Trainer memindahkan model ke GPU otomatis saat
`train()` dipanggil, tapi forward manual di luar Trainer tidak dapat pemindahan itu.
Tanpa baris ini muncul `RuntimeError: Expected all tensors to be on the same device`
persis di `embed_tokens` — weight masih di CPU sementara `input_ids` sudah di GPU.


In [ ]:
# ============================================================
# 15. PRE-FLIGHT FORWARD TEST
# ============================================================
print("=" * 70)
print("PRE-FLIGHT FORWARD TEST")
print("=" * 70)

sample_batch = guard_collator([train_ds[0]])

# FIX: pindahkan MODEL ke GPU, bukan cuma input.
if torch.cuda.is_available():
    guard_model = guard_model.cuda()
    sample_batch = {k: v.cuda() for k, v in sample_batch.items()}

guard_model.eval()
with torch.no_grad():
    sample_output = guard_model(**sample_batch)

print("Forward test  : SUCCESS")
print("Loss          :", sample_output["loss"].item())
print("Injection     :", tuple(sample_output["injection"].shape))
print("Shell         :", tuple(sample_output["shell"].shape))
print("Action        :", tuple(sample_output["action"].shape))

del sample_batch, sample_output
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# 16. TRAINING ARGUMENTS
# ============================================================
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

training_args = TrainingArguments(
    output_dir="./lora_adapter",

    # MEMORY: batch 1 x accum 16 = effective batch 16
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,

    # PRECISION
    bf16=use_bf16,
    fp16=not use_bf16,

    # TRAINING
    num_train_epochs=3,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",

    # LOGGING
    logging_steps=20,
    logging_first_step=True,
    report_to="none",

    # EVAL & SAVE
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,

    dataloader_num_workers=0,
    remove_unused_columns=False,   # WAJIB untuk model kustom
    optim="adamw_torch",
)

trainer = Trainer(
    model=guard_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=guard_collator,
)

print("Trainer n_gpu :", trainer.args.n_gpu, "(harus 1)")
print("Trainer device:", trainer.args.device)

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        a = torch.cuda.memory_allocated(i) / 1024**3
        r = torch.cuda.memory_reserved(i) / 1024**3
        t = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f"GPU {i}: {a:.2f} GB alloc | {r:.2f} GB reserved | {t:.2f} GB total")

In [ ]:
# ============================================================
# 17. TRAIN
# ============================================================
print("=" * 70)
print("START TRAINING")
print("=" * 70)

train_result = trainer.train()

print()
print("=" * 70)
print("TRAINING SELESAI")
print("=" * 70)
print(train_result)

In [ ]:
# ============================================================
# 18. SAVE
# ============================================================
OUTPUT_DIR = "./lora_adapter"

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved to:", OUTPUT_DIR)

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        a = torch.cuda.memory_allocated(i) / 1024**3
        r = torch.cuda.memory_reserved(i) / 1024**3
        print(f"GPU {i}: {a:.2f} GB alloc | {r:.2f} GB reserved")

## 19. Evaluasi in-training

Metrik cepat langsung dari model PyTorch. Ini **bukan** angka produksi — angka
produksi diukur lewat IR OpenVINO di NPU (`02_verify_export_deploy.ipynb`, atau
`scripts/eval_final.py`), yang juga melaporkan macro-F1 per kelas, ROC-AUC, dan
sweep threshold.


In [ ]:
# ============================================================
# 20. EVALUASI
# ============================================================
def eval_metrics(model, ds, n=500, label="eval"):
    dl = DataLoader(ds, batch_size=8, collate_fn=guard_collator, shuffle=False)

    preds_inj, true_inj = [], []
    preds_shell, true_shell = [], []
    preds_act, true_act = [], []

    model.eval()
    device = next(model.parameters()).device

    with torch.no_grad():
        for batch in dl:
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)

            preds_inj += out["injection"].float().cpu().tolist()
            true_inj += batch["inj_target"].float().cpu().tolist()

            preds_shell += out["shell"].float().cpu().tolist()
            true_shell += batch["shell_target"].float().cpu().tolist()

            preds_act += out["action"].argmax(-1).cpu().tolist()
            true_act += batch["action_target"].cpu().tolist()

            if len(preds_inj) >= n:
                break

    preds_inj, true_inj = preds_inj[:n], true_inj[:n]
    preds_shell, true_shell = preds_shell[:n], true_shell[:n]
    preds_act, true_act = preds_act[:n], true_act[:n]

    rmse_inj = np.sqrt(np.mean((np.array(preds_inj) - np.array(true_inj)) ** 2))
    rmse_shell = np.sqrt(np.mean((np.array(preds_shell) - np.array(true_shell)) ** 2))
    acc_act = np.mean(np.array(preds_act) == np.array(true_act))

    print(f"[{label}] RMSE inj: {rmse_inj:.3f} | "
          f"RMSE shell (PROXY): {rmse_shell:.3f} | "
          f"ACC action: {acc_act:.3f}")

    return {"rmse_inj": rmse_inj, "rmse_shell": rmse_shell, "acc_action": acc_act}


metrics_val = eval_metrics(guard_model, val_ds, label="validation")
metrics_test = eval_metrics(guard_model, test_ds, label="test (held-out)")

print("\nValidation:", metrics_val)
print("Test      :", metrics_test)

## 21. Langkah berikutnya

1. **Export ke OpenVINO IR** — `seq_len` **wajib 128** (== `MAX_LENGTH` di atas):
   ```bash
   python scripts/export_guard_ov.py --seq-len 128
   ```
   Jangan pakai `torch.jit.trace` atau `ov.convert_model(model, example_input=…)` —
   keduanya gagal pada Qwen2Model. Jalur yang berhasil ada di
   `references/npu-export-pitfalls.md`.

2. **Buktikan NPU yang mengeksekusi:**
   ```bash
   python scripts/prove_npu.py --npu-only
   ```

3. **Eval produksi lewat IR/NPU:**
   ```bash
   python scripts/eval_final.py --device NPU
   ```

4. **Deploy:** `INIZ_GUARD_DEVICE=NPU python guard_server.py`

Semuanya dirangkai di `02_verify_export_deploy.ipynb`.
